# Notebook VI — Economic Suitability Analysis
<hr>

> **BETA** — This module is under active development. Cost and price parameters below are placeholder values and must be updated with current Pakistan market data before interpreting results.

## Overview
This notebook runs **Module VI** — the final step of the PyAEZ pipeline. It performs a break-even analysis to determine whether a crop is economically viable at each pixel, given production costs, achievable yields, and farm-gate price scenarios.

### Method
Based on the NAEZ Thailand (2016) framework:
```
Net Revenue (PKR/ha) = Farm Price (PKR/ton) × Yield (ton/ha) − Production Cost (PKR/ha)
```
The analysis is run across a range of farm-gate prices to account for market variability.

### Prerequisites
NB1 through NB5 must have been run first. This notebook reads:
- `data_output/NB5/terr_soil_clim_adj_yield_maiz_rain.tif`
- `data_output/NB5/terr_soil_clim_adj_yield_maiz_irr.tif`

### What this notebook produces (saved to `data_output/NB6/`)
| Output file | Description |
|---|---|
| `maiz_rain_net_revenue.tif` | Mean net revenue — rainfed (PKR/ha) |
| `maiz_irr_net_revenue.tif` | Mean net revenue — irrigated (PKR/ha) |
| `maiz_rain_net_revenue_class.tif` | Classified net revenue — rainfed (1–5) |
| `maiz_irr_net_revenue_class.tif` | Classified net revenue — irrigated (1–5) |
| `maiz_rain_norm_net_revenue.tif` | Normalised net revenue — rainfed (0–1) |
| `maiz_irr_norm_net_revenue.tif` | Normalised net revenue — irrigated (0–1) |

Prepared by Geoinformatics Center, AIT — Pakistan adaptation by Saqib Ali
<hr>

### Step 1 — Connect to Google Drive (Colab only)

Mount Google Drive to access the repo at `/content/drive/MyDrive/PK-PyAEZ/`.

> **Skip** if running locally.

In [ ]:
# ── Google Colab setup ──────────────────────────────────────────────
# Run these steps ONLY when using Google Colab.

# Step 1: Mount Google Drive to access the repo
# from google.colab import drive
# drive.mount('/content/drive')

### Step 2 — Install dependencies (Colab only)

Installs GDAL and all required packages, then installs `pyaez` in editable mode from the Drive copy.

> **Skip** if running locally with the conda environment already set up.

In [ ]:
import sys

if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run(['apt-get', 'install', '-y', '-q', 'gdal-bin', 'libgdal-dev', 'python3-gdal'],
                   capture_output=True)
    _ver = subprocess.check_output(['gdal-config', '--version']).decode().strip()
    subprocess.run(['pip', 'install', '-q',
                    f'GDAL=={_ver}', 'numba', 'openpyxl', 'rasterio', 'requests', 'scipy', 'pandas'],
                   capture_output=True)
    subprocess.run(['pip', 'install', '-q', '-e', '/content/drive/MyDrive/PK-PyAEZ'], capture_output=True)
    print(f"Colab setup complete — GDAL {_ver}")

### Step 3 — Import Python libraries

| Library | Purpose |
|---|---|
| `numpy` | Array operations on yield and revenue maps |
| `matplotlib` | Plots net revenue and suitability maps |
| `gdal` (osgeo) | Opens NB5 yield GeoTIFFs; saves output rasters |
| `os`, `sys` | File path and module path management |

`gdal.UseExceptions()` silences the GDAL 4.0 FutureWarning.

In [ ]:
'''import supporting libraries'''
import matplotlib.pyplot as plt
import numpy as np
import os
try:
    from osgeo import gdal
except:
    import gdal
gdal.UseExceptions()
import sys

### Step 4 — Set working directory

Sets CWD to the repo root so all `./data_input/` and `./data_output/` paths resolve correctly.

- **Colab**: `/content/drive/MyDrive/PK-PyAEZ`
- **Local**: `..` (one level up from `tutorials/`)

In [ ]:
'Set the working directory'
import sys, os

if 'google.colab' in sys.modules:
    work_dir = '/content/drive/MyDrive/PK-PyAEZ'
else:
    work_dir = '..'

os.chdir(work_dir)
sys.path.insert(0, os.getcwd())
os.getcwd()

### Step 5 — Create output folder

Creates `data_output/NB6/` if it does not already exist. All net revenue rasters are saved there.

In [ ]:
import os
folder_path = './data_output/NB6/'
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print("Folder created successfully.")
else:
    print("Folder already exists.")

<hr>

## MODULE VI — Economic Suitability Analysis

### Step 6 — Set economic parameters

> **⚠ Pakistan-specific values required.** The defaults below are illustrative. Update with current data from MNFSR, PBS, or commodity exchange reports.

#### Production costs (`cost_maize`) — PKR/ha at 4 management levels
| Index | Management level | Cost (PKR/ha) |
|---|---|---|
| 0 | Subsistence | 85,000 |
| 1 | Low input | 80,000 |
| 2 | Intermediate | 72,000 |
| 3 | High input | 65,000 |

#### Reference yields (`yield_maize`) — ton/ha at 4 management levels
Corresponds to the same 4 management levels as costs above.

#### Farm-gate price range (`farm_price_maize`) — PKR/ton
A range of prices is used to capture market variability. Net revenue is averaged across all price scenarios.

#### Yield maps (from NB5) — converted from kg/ha to ton/ha (divide by 1000)

In [ ]:
'''preparing the data for maize'''

cost_maize = np.array([85000, 80000, 72000, 65000])  # PKR/ha — update with current values
yield_maize = np.array([4.8, 5.5, 6.1, 8.7]) / 2     # ton/ha — divided by 2 as conservative estimate
farm_price_maize = np.arange(40000, 65000, 5000)       # PKR/ton — update with current values

### Load NB5 outputs and convert kg/ha → ton/ha
yield_rain_maize = (gdal.Open(r'./data_output/NB5/terr_soil_clim_adj_yield_maiz_rain.tif').ReadAsArray()) / 1000
yield_irr_maize  = (gdal.Open(r'./data_output/NB5/terr_soil_clim_adj_yield_maiz_irr.tif').ReadAsArray())  / 1000

### Step 7 — Initialise EconomicSuitability and register crops

`addACrop()` registers a crop with its full economic profile. Call it once per crop and water-supply scenario.

| Parameter | Type | Description |
|---|---|---|
| `crop_name` | str | Unique identifier used to retrieve results |
| `crop_cost` | array | Production costs (PKR/ha) at each management level |
| `crop_yield` | array | Reference yields (ton/ha) at each management level |
| `farm_price` | array | Range of farm-gate prices (PKR/ton) to simulate |
| `yield_map` | 2D array | Pixel-wise yield from NB5 (ton/ha) |

Multiple crops can be registered and compared, but each `crop_name` must be unique.

In [ ]:
'''importing library and registering crops'''

from pyaez import EconomicSuitability
econ_su = EconomicSuitability.EconomicSuitability()

from pyaez import UtilitiesCalc
obj_utilities = UtilitiesCalc.UtilitiesCalc()

econ_su.addACrop(crop_name='maize_rain', crop_cost=cost_maize, crop_yield=yield_maize,
                 farm_price=farm_price_maize, yield_map=yield_rain_maize)
econ_su.addACrop(crop_name='maize_irr',  crop_cost=cost_maize, crop_yield=yield_maize,
                 farm_price=farm_price_maize, yield_map=yield_irr_maize)

### Step 8 — Compute net revenue outputs

Three output types are computed for each registered crop:

| Method | Output | Description |
|---|---|---|
| `getNetRevenue()` | PKR/ha | Mean net revenue averaged across all farm-price scenarios |
| `getClassifiedNetRevenue()` | 1–5 class | Revenue suitability class (1 = not suitable → 5 = very suitable) |
| `getNormalizedNetRevenue()` | 0–1 | Net revenue normalised by the maximum across all pixels |

Negative net revenue → crop is not economically viable at that pixel under current assumptions.

In [ ]:
'''compute results for rainfed and irrigated'''

maize_rev_rain       = econ_su.getNetRevenue('maize_rain')            # PKR/ha
maize_rev_rain_class = econ_su.getClassifiedNetRevenue('maize_rain')
maize_rev_rain_norm  = econ_su.getNormalizedNetRevenue('maize_rain')

maize_rev_irr        = econ_su.getNetRevenue('maize_irr')             # PKR/ha
maize_rev_irr_class  = econ_su.getClassifiedNetRevenue('maize_irr')
maize_rev_irr_norm   = econ_su.getNormalizedNetRevenue('maize_irr')

### Step 9 — Visualise net revenue (PKR/ha)

Side-by-side comparison of rainfed vs irrigated mean net revenue. Both maps share the same colour scale for direct comparison. Negative values (blue) indicate economic loss under current price assumptions.

In [ ]:
'''visualize net revenue'''
plt.figure(1, figsize=(15, 8))

plt.subplot(1, 2, 1)
plt.imshow(maize_rev_rain, vmax=np.max([maize_rev_rain, maize_rev_irr]))
plt.colorbar(shrink=0.8)
plt.title('Maize Rainfed — Net Revenue (PKR/ha)')

plt.subplot(1, 2, 2)
plt.imshow(maize_rev_irr, vmax=np.max([maize_rev_rain, maize_rev_irr]))
plt.colorbar(shrink=0.8)
plt.title('Maize Irrigated — Net Revenue (PKR/ha)')

### Step 10 — Visualise classified net revenue

Revenue suitability classes (same scale for both maps):

| Class | Description |
|---|---|
| 1 | Not suitable — net loss |
| 2 | Marginally suitable |
| 3 | Moderately suitable |
| 4 | Suitable |
| 5 | Very suitable — high net revenue |

In [ ]:
'''visualize classified net revenue'''
plt.figure(1, figsize=(15, 8))

plt.subplot(1, 2, 1)
plt.imshow(maize_rev_rain_class, vmax=np.max([maize_rev_rain_class, maize_rev_irr_class]))
plt.colorbar(shrink=0.8)
plt.title('Maize Rainfed — Classified Net Revenue')

plt.subplot(1, 2, 2)
plt.imshow(maize_rev_irr_class, vmax=np.max([maize_rev_rain_class, maize_rev_irr_class]))
plt.colorbar(shrink=0.8)
plt.title('Maize Irrigated — Classified Net Revenue')

### Step 11 — Visualise normalised net revenue (0–1)

Normalised maps allow cross-crop comparison on a common scale (0 = minimum, 1 = maximum revenue pixel).

In [ ]:
'''visualize normalized net revenue'''
plt.figure(1, figsize=(15, 8))

plt.subplot(1, 2, 1)
plt.imshow(maize_rev_rain_norm, vmax=np.max([maize_rev_rain_norm, maize_rev_irr_norm]))
plt.colorbar(shrink=0.8)
plt.title('Maize Rainfed — Normalised Net Revenue')

plt.subplot(1, 2, 2)
plt.imshow(maize_rev_irr_norm, vmax=np.max([maize_rev_rain_norm, maize_rev_irr_norm]))
plt.colorbar(shrink=0.8)
plt.title('Maize Irrigated — Normalised Net Revenue')

### Step 12 — Save all outputs

| File | Content |
|---|---|
| `maiz_rain/irr_net_revenue.tif` | Mean net revenue (PKR/ha) |
| `maiz_rain/irr_net_revenue_class.tif` | Revenue suitability class (1–5) |
| `maiz_rain/irr_norm_net_revenue.tif` | Normalised net revenue (0–1) |

In [ ]:
# Saving all outputs
obj_utilities.saveRaster(r'./data_input/PK_Admin.tif', r'./data_output/NB6/maiz_rain_net_revenue.tif',       maize_rev_rain)
obj_utilities.saveRaster(r'./data_input/PK_Admin.tif', r'./data_output/NB6/maiz_irr_net_revenue.tif',        maize_rev_irr)

obj_utilities.saveRaster(r'./data_input/PK_Admin.tif', r'./data_output/NB6/maiz_rain_net_revenue_class.tif', maize_rev_rain_class)
obj_utilities.saveRaster(r'./data_input/PK_Admin.tif', r'./data_output/NB6/maiz_irr_net_revenue_class.tif',  maize_rev_irr_class)

obj_utilities.saveRaster(r'./data_input/PK_Admin.tif', r'./data_output/NB6/maiz_rain_norm_net_revenue.tif',  maize_rev_rain_norm)
obj_utilities.saveRaster(r'./data_input/PK_Admin.tif', r'./data_output/NB6/maiz_irr_norm_net_revenue.tif',   maize_rev_irr_norm)

<hr>

### END OF MODULE VI — Economic Suitability Analysis

This completes the full **6-notebook PyAEZ pipeline** for Pakistan.

#### Full pipeline summary
| Notebook | Module | Output |
|---|---|---|
| NB1 | Climate Regime | LGP, thermal zones, ETo |
| NB2 | Crop Simulation | Max attainable yield (fc1, fc2) |
| NB3 | Climatic Constraints | fc3-adjusted yield |
| NB4 | Soil Constraints | fc4-adjusted yield |
| NB5 | Terrain Constraints | fc5-adjusted yield |
| NB6 | Economic Suitability | Net revenue maps |

<hr>